# 04 — Train: Ridge regression

Searches Ridge regularization for the configured target station's direct 24-hour water-level forecast over the joined feature artifacts, then evaluates the selected model once on the sealed test cohort.

**Inputs:** joined train/test feature artifacts and their metadata contract  
**Outputs:** in-notebook prediction preview/test metrics, an MLflow run hierarchy, and the selected model plus manifest in `models/`

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and loads the joined feature metadata. Its predictor columns are the source of truth for the model inputs: target-station engineered features plus raw measurements from every retained station at issue time `t`. The Ridge alpha search and validation policy are explicit constants so every fold and MLflow run remains inspectable.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed/joined` | Directory the joined Stage-3 Parquets and metadata are read from. |
| `PREDICTION_PREVIEW_ROWS` | `5` | Number of scored test rows shown in the final preview. |
| `FULL_FEATURE_COLUMNS` | all metadata-declared predictors | The complete predictor contract used for common eligibility; raw timestamps and metadata fields are not model inputs. |
| `FEATURE_SUBSETS` | six predefined subsets | Candidate feature lists derived from the full metadata contract in metadata order. |
| `TARGET_COLUMNS` | `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` | The 24 future water levels predicted directly from one issue-time feature vector. |
| `FORECAST_HORIZON_HOURS` | `24` | Number of direct future target outputs and the metadata contract width. |
| `RIDGE_ALPHAS` | `[0.01, 0.1, 1.0, 10.0, 100.0]` | Candidate L2 regularization strengths. |
| `N_VALIDATION_FOLDS` | `5` | Number of expanding-window validation folds. |
| `INITIAL_TRAIN_FRACTION` | `0.50` | Approximate fraction of eligible rows in the first fold's training window. |
| `EMBARGO_HOURS` | `24` | Number of rows left between each fold's training and validation windows. |
| `CV_SELECTION_METRIC` | `"rmse"` | Aggregate CV metric used to select alpha; `"mae"` is also supported. |
| `MLFLOW_EXPERIMENT_NAME` | `"ridge"` | Experiment receiving the parent, nested fold, and final test runs. |
| `MODEL_PATH` | `models/ridge_{TARGET_STATION_ID}.joblib` | Bundled scaler and selected Ridge estimator trained on all eligible training rows. |
| `MODEL_METADATA_PATH` | `models/ridge_{TARGET_STATION_ID}.json` | Reproducibility manifest containing the feature contract, selected subset and alpha, CV results, and training range. |

## Joint feature-subset and alpha search

The notebook compares one global `(feature subset, alpha)` pair with five expanding-window folds. The six predefined subsets are derived from metadata-declared predictors and retain their metadata order:

| Subset | Intended predictors | Current size |
| --- | --- | ---: |
| `full` | All declared predictors | 81 |
| `all_station_hydrology_quality_time` | Water-level history, imputation indicators, and calendar signals for every station; excludes weather | 55 |
| `raw_all_stations` | Current `water_level`, `imputed`, precipitation, and temperature for every station | 32 |
| `target_station_full` | All declared predictors for the target station only | 53 |
| `target_station_hydrology_quality_time` | Target-station water-level history, imputation indicators, and calendar signals | 41 |
| `current_water_levels_all_stations` | Current `water_level` for every station | 8 |

The full contract determines eligibility once for both artifacts. Consequently, all candidates use the same 48,403 training rows, 15,196 sealed-test rows, and identical fold indices in the current data. A missing predictor excluded by a candidate still removes that timestamp for every candidate; smaller subsets therefore do not gain additional eligible rows in this controlled ablation. The search performs `6 × 5 × 5 = 150` fold fits, then retrains only the selected candidate and evaluates the sealed test once.

In [ ]:
import hashlib
import json
from pathlib import Path
from uuid import uuid4

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import dump
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    FORECAST_HORIZON_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.metrics import metric_tables
from src.training import (
    absolute_error_boxplot_payload,
    build_feature_subsets,
    cv_error_boxplot_payload,
    error_boxplots_figure,
    load_joined_training_data,
    numeric_predictors,
    predicted_vs_actual_figure,
    prediction_preview,
    prepare_model_rows,
    summarize_cv_metrics,
    time_series_splits,
    validate_predictions,
)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
NOTEBOOK_EXECUTION_UUID = str(uuid4())
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
MLFLOW_EXPERIMENT_NAME = "ridge"
PREDICTION_PREVIEW_ROWS = 5
if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")
station_id = TARGET_STATION_ID
MODEL_DIR = Path("models")
MODEL_PATH = MODEL_DIR / f"ridge_{station_id}.joblib"
MODEL_METADATA_PATH = MODEL_DIR / f"ridge_{station_id}.json"


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as input_file:
        for chunk in iter(lambda: input_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## Shared evaluation cohort

The model is fit and scored on rows from the joined feature artifacts. One row is one timestamp `t`, and it qualifies only when both conditions hold:

1. **Stage 3 marked the future window valid.** `{TARGET_STATION_ID}__target_valid` is true, and all 24 `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` values are present.
2. **Every model input is present.** All full-contract predictors must be available: the target station's engineered features plus every retained station's raw water level, imputation flag, precipitation, and temperature at issue time `t`.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model — not even a scaler mean — is ever computed from it. Eligibility is deliberately based on `FULL_FEATURE_COLUMNS`, not a candidate subset, so all 30 candidates compare the same cohort.

## Shared helpers

Joined-contract loading, common-cohort preparation, ordered feature subsets, chronological folds, prediction checks, metric summaries, previews, and evaluation figures come from `src.training`. Ridge candidate ranking remains local because its subset/alpha tie-breaking policy is estimator-specific.

In [ ]:
def select_candidate(
    cv_results: pd.DataFrame, metric: str = "rmse"
) -> tuple[str, float]:
    """Select one subset/alpha pair with deterministic tie-breaking."""
    if metric not in {"mae", "rmse"}:
        raise ValueError("CV selection metric must be either 'mae' or 'rmse'")
    metric_column = f"{metric}_mean"
    required_columns = {"subset", "alpha", "feature_count", metric_column}
    missing = sorted(required_columns.difference(cv_results.columns))
    if missing or cv_results.empty:
        raise ValueError(f"CV results are empty or missing columns: {missing}")
    if cv_results[[metric_column, "alpha", "feature_count"]].isna().any().any():
        raise ValueError("CV candidate results contain null ranking values")
    ranked = cv_results.sort_values(
        [metric_column, "feature_count", "alpha", "subset"],
        kind="stable",
    )
    winner = ranked.iloc[0]
    return str(winner["subset"]), float(winner["alpha"])

## Load joined feature artifacts

Loads the joined feature metadata, `all_stations_train_features.parquet`, and `all_stations_test_features.parquet` from the Stage-3 directory. Their station, horizon, and column contracts are checked before the fit, so a missing or incompatible artifact fails before any model work begins.

In [ ]:
contract, train_features, test_features = load_joined_training_data(
    METADATA_PATH,
    train_path,
    test_path,
    station_id=station_id,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
)
TARGET_COLUMNS = list(contract.target_columns)
FULL_FEATURE_COLUMNS = list(contract.predictor_columns)
FEATURE_SUBSETS = build_feature_subsets(
    contract,
    weather_variables=WEATHER_VARIABLES,
)
INPUT_PARQUET_SHA256_PARAMS = {
    "train_input_sha256": _sha256_file(train_path),
    "test_input_sha256": _sha256_file(test_path),
}

## Apply the eligibility cohort

Prepares the train and test cohorts independently with `prepare_model_rows()`. Each cohort keeps only target-valid rows with complete predictors and targets, then sorts them chronologically. If either split has no eligible row, the notebook stops rather than fitting on an empty frame or reporting a metric computed from nothing.

In [ ]:
train_rows = prepare_model_rows(
    train_features,
    contract,
    artifact_name="train",
)
test_rows = prepare_model_rows(
    test_features,
    contract,
    artifact_name="test",
)

## Joint time-series subset and alpha search

Eligible training rows are sorted by issue time before `TimeSeriesSplit` creates five expanding-window folds. The explicit `test_size` allocates the post-initial-training portion across the folds, while the 24-row gap acts as the requested hourly embargo. Each `(subset, alpha)` candidate has one MLflow parent and each fold has one nested child run: `6 × 5 × 5 = 150` fits. The current execution must produce the complete 30-candidate Cartesian product before selection.

Every fold fits its own `StandardScaler` and 24-output `Ridge` model using only that fold's training rows and the candidate's explicit columns. The sealed test cohort is not referenced until the final fit below.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_splitter, cv_splits, validation_test_size = time_series_splits(
    len(train_rows),
    initial_train_fraction=INITIAL_TRAIN_FRACTION,
    n_validation_folds=N_VALIDATION_FOLDS,
    embargo_rows=EMBARGO_HOURS,
)
cv_results_rows = []
cv_horizon_rows_by_candidate = {}
expected_candidate_keys = {
    (subset_name, float(alpha))
    for subset_name in FEATURE_SUBSETS
    for alpha in RIDGE_ALPHAS
}

for subset_name, feature_columns in FEATURE_SUBSETS.items():
    for alpha in RIDGE_ALPHAS:
        fold_aggregate_rows = []
        fold_horizon_rows = []
        with mlflow.start_run(
            run_name=f"ridge_cv_{subset_name}_{alpha:g}",
            nested=False,
            tags={
                "phase": "cv",
                "run_type": "candidate_parent",
                "subset": subset_name,
                "execution_uuid": NOTEBOOK_EXECUTION_UUID,
            },
        ):
            mlflow.log_params(
                {
                    "phase": "cv",
                    "run_type": "candidate_parent",
                    "subset": subset_name,
                    "feature_count": len(feature_columns),
                    "feature_columns": json.dumps(feature_columns),
                    **INPUT_PARQUET_SHA256_PARAMS,
                    "alpha": alpha,
                    "n_validation_folds": N_VALIDATION_FOLDS,
                    "validation_test_size": validation_test_size,
                    "embargo_hours": EMBARGO_HOURS,
                    "selection_metric": CV_SELECTION_METRIC,
                    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                    "common_train_rows": len(train_rows),
                }
            )

            for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
                cv_splits, start=1
            ):
                fold_train_rows = train_rows.iloc[fold_train_indices]
                fold_validation_rows = train_rows.iloc[fold_validation_indices]
                fold_scaler = StandardScaler()
                fold_train_predictors = fold_scaler.fit_transform(
                    numeric_predictors(fold_train_rows, feature_columns)
                )
                fold_validation_predictors = fold_scaler.transform(
                    numeric_predictors(fold_validation_rows, feature_columns)
                )
                fold_ridge = Ridge(alpha=alpha)
                fold_ridge.fit(fold_train_predictors, fold_train_rows[TARGET_COLUMNS])
                fold_predictions = validate_predictions(
                    fold_ridge.predict(fold_validation_predictors),
                    expected_rows=len(fold_validation_rows),
                    target_columns=TARGET_COLUMNS,
                    artifact_name="fold",
                )

                fold_aggregate, fold_per_horizon = metric_tables(
                    fold_validation_rows[TARGET_COLUMNS],
                    fold_predictions,
                    target_columns=TARGET_COLUMNS,
                    station_id=station_id,
                )
                fold_aggregate_rows.append(fold_aggregate.iloc[0])
                fold_horizon_rows.append(fold_per_horizon)
                with mlflow.start_run(
                    run_name=f"ridge_cv_{subset_name}_{alpha:g}_fold_{fold_number}",
                    nested=True,
                    tags={
                        "phase": "cv",
                        "run_type": "fold",
                        "subset": subset_name,
                        "fold": str(fold_number),
                        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
                    },
                ):
                    mlflow.log_params(
                        {
                            "phase": "cv",
                            "run_type": "fold",
                            "subset": subset_name,
                            "feature_count": len(feature_columns),
                            **INPUT_PARQUET_SHA256_PARAMS,
                            "alpha": alpha,
                            "fold": fold_number,
                            "train_rows": len(fold_train_rows),
                            "validation_rows": len(fold_validation_rows),
                            "gap_rows": EMBARGO_HOURS,
                            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                            "train_start": fold_train_rows["timestamp"]
                            .iloc[0]
                            .isoformat(),
                            "train_end": fold_train_rows["timestamp"]
                            .iloc[-1]
                            .isoformat(),
                            "validation_start": fold_validation_rows["timestamp"]
                            .iloc[0]
                            .isoformat(),
                            "validation_end": fold_validation_rows["timestamp"]
                            .iloc[-1]
                            .isoformat(),
                            "train_index_start": int(fold_train_indices[0]),
                            "train_index_end": int(fold_train_indices[-1]),
                            "validation_index_start": int(fold_validation_indices[0]),
                            "validation_index_end": int(fold_validation_indices[-1]),
                        }
                    )
                    mlflow.log_metrics(
                        {
                            "fold_mae": float(fold_aggregate.iloc[0]["mae"]),
                            "fold_rmse": float(fold_aggregate.iloc[0]["rmse"]),
                            "fold_me": float(fold_aggregate.iloc[0]["me"]),
                            "fold_r2": float(fold_aggregate.iloc[0]["r2"]),
                            **{
                                f"fold_mae_horizon_{row.horizon_hours:02d}": float(
                                    row.mae
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_me_horizon_{row.horizon_hours:02d}": float(
                                    row.me
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_r2_horizon_{row.horizon_hours:02d}": float(
                                    row.r2
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                            **{
                                f"fold_rmse_horizon_{row.horizon_hours:02d}": float(
                                    row.rmse
                                )
                                for row in fold_per_horizon.itertuples()
                            },
                        }
                    )

            fold_aggregate_metrics = pd.DataFrame(fold_aggregate_rows)
            fold_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
            parent_metrics = summarize_cv_metrics(
                fold_aggregate_metrics,
                fold_horizon_metrics,
            )
            candidate_key = (subset_name, float(alpha))
            cv_horizon_rows_by_candidate[candidate_key] = fold_horizon_rows.copy()
            mlflow.log_metrics(parent_metrics)
            cv_results_rows.append(
                {
                    "subset": subset_name,
                    "feature_count": len(feature_columns),
                    "alpha": float(alpha),
                    "mae_mean": parent_metrics["cv_mae_mean"],
                    "mae_std": parent_metrics["cv_mae_std"],
                    "rmse_mean": parent_metrics["cv_rmse_mean"],
                    "rmse_std": parent_metrics["cv_rmse_std"],
                    "me_mean": parent_metrics["cv_me_mean"],
                    "me_std": parent_metrics["cv_me_std"],
                    "r2_mean": parent_metrics["cv_r2_mean"],
                    "r2_std": parent_metrics["cv_r2_std"],
                    **{
                        metric_name: metric_value
                        for metric_name, metric_value in parent_metrics.items()
                        if metric_name
                        not in {
                            "cv_mae_mean",
                            "cv_mae_std",
                            "cv_rmse_mean",
                            "cv_rmse_std",
                            "cv_me_mean",
                            "cv_me_std",
                            "cv_r2_mean",
                            "cv_r2_std",
                        }
                    },
                }
            )

cv_experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if cv_experiment is None:
    raise ValueError(f"MLflow experiment {MLFLOW_EXPERIMENT_NAME!r} was not found")
current_cv_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'candidate_parent'"
    ),
)
current_fold_runs = mlflow.search_runs(
    experiment_ids=[cv_experiment.experiment_id],
    filter_string=(
        f"tags.execution_uuid = '{NOTEBOOK_EXECUTION_UUID}' "
        "and tags.phase = 'cv' "
        "and tags.run_type = 'fold'"
    ),
)
parent_keys = {
    (str(row["tags.subset"]), float(row["params.alpha"]))
    for _, row in current_cv_runs.iterrows()
}
if (
    len(current_cv_runs) != len(expected_candidate_keys)
    or parent_keys != expected_candidate_keys
):
    raise ValueError(
        "Current execution must produce the complete 30-candidate subset/alpha product: "
        f"expected {len(expected_candidate_keys)} {sorted(expected_candidate_keys)}, "
        f"got {len(current_cv_runs)} {sorted(parent_keys)}"
    )
expected_fold_count = len(expected_candidate_keys) * N_VALIDATION_FOLDS
if len(current_fold_runs) != expected_fold_count:
    raise ValueError(
        f"Current execution must produce {expected_fold_count} nested fold runs, got {len(current_fold_runs)}"
    )
fold_keys = {
    (
        str(row["tags.subset"]),
        float(row["params.alpha"]),
        int(row["tags.fold"]),
    )
    for _, row in current_fold_runs.iterrows()
}
expected_fold_keys = {
    (subset_name, float(alpha), fold_number)
    for subset_name, alpha in expected_candidate_keys
    for fold_number in range(1, N_VALIDATION_FOLDS + 1)
}
if fold_keys != expected_fold_keys:
    raise ValueError(
        "Current execution fold runs do not cover every candidate and fold"
    )
cv_results = pd.DataFrame(cv_results_rows)
if len(cv_results) != len(expected_candidate_keys):
    raise ValueError("The in-memory CV result table is incomplete")
if set(zip(cv_results["subset"], cv_results["alpha"])) != expected_candidate_keys:
    raise ValueError(
        "The in-memory CV result table does not match the candidate product"
    )
cv_results = cv_results.sort_values(["subset", "alpha"], kind="stable").reset_index(
    drop=True
)
selected_subset, selected_alpha = select_candidate(cv_results, CV_SELECTION_METRIC)
selected_feature_columns = FEATURE_SUBSETS[selected_subset]
fold_horizon_rows = cv_horizon_rows_by_candidate[(selected_subset, selected_alpha)]
print(
    f"Selected Ridge candidate by CV {CV_SELECTION_METRIC.upper()}: "
    f"{selected_subset!r}, alpha={selected_alpha:g}"
)
display(
    cv_results[
        [
            "subset",
            "feature_count",
            "alpha",
            "mae_mean",
            "mae_std",
            "rmse_mean",
            "rmse_std",
            "me_mean",
            "me_std",
            "r2_mean",
            "r2_std",
        ]
    ]
)

## Retrain the selected subset and alpha

The selected `(feature subset, alpha)` pair is retrained once on all eligible, chronologically ordered training rows. The scaler and Ridge estimator are bundled in a single pipeline, fitted on the full eligible training cohort, and persisted with a reproducibility manifest before the sealed test predictors are scored.

In [ ]:
final_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=selected_alpha)),
    ]
)
final_model.fit(
    numeric_predictors(train_rows, selected_feature_columns),
    train_rows[TARGET_COLUMNS],
)
test_predictions = validate_predictions(
    final_model.predict(numeric_predictors(test_rows, selected_feature_columns)),
    expected_rows=len(test_rows),
    target_columns=TARGET_COLUMNS,
    artifact_name="test",
)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
dump(final_model, MODEL_PATH)
cv_results_records = []
for row in cv_results.to_dict(orient="records"):
    cv_results_records.append(
        {
            key: (
                str(value)
                if key == "subset"
                else int(value)
                if key == "feature_count"
                else float(value)
            )
            for key, value in row.items()
        }
    )
model_manifest = {
    "schema_version": "1.1",
    "model_path": str(MODEL_PATH),
    "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    "model_type": "sklearn.pipeline.Pipeline",
    "estimator": "Ridge",
    "preprocessor": "StandardScaler",
    "station_id": station_id,
    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
    "full_feature_columns": FULL_FEATURE_COLUMNS,
    "selected_subset": selected_subset,
    "feature_subset": selected_subset,
    "selected_feature_columns": selected_feature_columns,
    "feature_subsets": FEATURE_SUBSETS,
    "target_columns": TARGET_COLUMNS,
    "selected_alpha": float(selected_alpha),
    "selection_metric": CV_SELECTION_METRIC,
    "tie_breaking": [
        f"lowest aggregate CV {CV_SELECTION_METRIC.upper()}",
        "fewer features",
        "smaller alpha",
        "stable subset name",
    ],
    "cv_results": cv_results_records,
    "common_cohort_eligibility": {
        "contract": "full_feature_columns",
        "rule": "target_valid and complete full predictor and target contract",
        "train_raw_rows": len(train_features),
        "train_eligible_rows": len(train_rows),
        "test_raw_rows": len(test_features),
        "test_eligible_rows": len(test_rows),
        "same_folds_for_all_candidates": True,
    },
    "training": {
        "source_artifact": str(train_path),
        "raw_rows": len(train_features),
        "eligible_rows": len(train_rows),
        "eligibility": "target_valid and complete full predictor and target contract",
        "timestamp_start": train_rows["timestamp"].iloc[0].isoformat(),
        "timestamp_end": train_rows["timestamp"].iloc[-1].isoformat(),
    },
}
MODEL_METADATA_PATH.write_text(
    json.dumps(model_manifest, indent=2) + "\n", encoding="utf-8"
)
print(f"Saved Ridge model to {MODEL_PATH}")
print(f"Saved Ridge model manifest to {MODEL_METADATA_PATH}")

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort reports aggregate MAE/RMSE, the same metrics for each lead in the direct 24-hour forecast, and a short preview for comparison with actual targets. Plot and MLflow labels identify both the selected subset and alpha. There is no second pass and no refitting.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
if not np.isfinite(aggregate_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite aggregate metrics")
if not np.isfinite(per_horizon_metrics[["mae", "rmse", "me", "r2"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite horizon metrics")
with mlflow.start_run(
    run_name=f"ridge_test_{selected_subset}_alpha_{selected_alpha:g}",
    nested=False,
    tags={
        "phase": "test",
        "run_type": "sealed_test",
        "subset": selected_subset,
        "execution_uuid": NOTEBOOK_EXECUTION_UUID,
    },
):
    mlflow.log_params(
        {
            "phase": "test",
            "run_type": "sealed_test",
            "subset": selected_subset,
            "feature_count": len(selected_feature_columns),
            "feature_columns": json.dumps(selected_feature_columns),
            **INPUT_PARQUET_SHA256_PARAMS,
            "alpha": selected_alpha,
            "selection_metric": CV_SELECTION_METRIC,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
            "cv_selected_metric": float(
                cv_results.loc[
                    cv_results["subset"].eq(selected_subset)
                    & cv_results["alpha"].eq(selected_alpha),
                    f"{CV_SELECTION_METRIC}_mean",
                ].iloc[0]
            ),
            "scored_issue_times": len(test_rows),
        }
    )
    mlflow.log_metrics(
        {
            "test_mae": float(aggregate_metrics.iloc[0]["mae"]),
            "test_rmse": float(aggregate_metrics.iloc[0]["rmse"]),
            "test_me": float(aggregate_metrics.iloc[0]["me"]),
            "test_r2": float(aggregate_metrics.iloc[0]["r2"]),
            **{
                f"test_mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_me_horizon_{row.horizon_hours:02d}": float(row.me)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_r2_horizon_{row.horizon_hours:02d}": float(row.r2)
                for row in per_horizon_metrics.itertuples()
            },
            **{
                f"test_rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                for row in per_horizon_metrics.itertuples()
            },
        }
    )
    cv_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
    cv_boxplot_values, horizon_labels = cv_error_boxplot_payload(
        cv_horizon_metrics,
        target_columns=TARGET_COLUMNS,
    )
    cv_rmse_mae_boxplots_fig = error_boxplots_figure(
        cv_boxplot_values,
        horizon_labels,
        title=f"Ridge CV errors — {selected_subset}, alpha={selected_alpha:g}",
    )
    mlflow.log_figure(cv_rmse_mae_boxplots_fig, "cv_rmse_mae_boxplots.png")
    plt.show()
    plt.close(cv_rmse_mae_boxplots_fig)
    (
        test_boxplot_values,
        test_horizon_labels,
        test_summary_markers,
    ) = absolute_error_boxplot_payload(
        test_rows,
        test_predictions,
        per_horizon_metrics,
        TARGET_COLUMNS,
    )
    test_error_boxplots_fig = error_boxplots_figure(
        test_boxplot_values,
        test_horizon_labels,
        title=f"Ridge final-test errors — {selected_subset}, alpha={selected_alpha:g}",
        x_axis_label="Forecast horizon",
        summary_markers=test_summary_markers,
    )
    mlflow.log_figure(test_error_boxplots_fig, "test_error_boxplots.png")
    plt.show()
    plt.close(test_error_boxplots_fig)
    test_predicted_vs_actual_fig = predicted_vs_actual_figure(
        test_rows[TARGET_COLUMNS],
        test_predictions,
        TARGET_COLUMNS,
        title=f"Ridge predicted vs actual — {selected_subset}, alpha={selected_alpha:g}",
    )
    mlflow.log_figure(test_predicted_vs_actual_fig, "test_predicted_vs_actual.png")
    plt.show()
    plt.close(test_predicted_vs_actual_fig)
print(
    f"Ridge test results for {station_id} "
    f"(selected subset={selected_subset!r}, alpha={selected_alpha:g})"
)
display(aggregate_metrics)
display(per_horizon_metrics)
display(
    prediction_preview(
        test_rows,
        test_predictions,
        target_columns=TARGET_COLUMNS,
    ).head(PREDICTION_PREVIEW_ROWS)
)

# Ridge MLflow candidate comparison

This read-only section queries the Ridge experiment directly from MLflow. It compares the logged feature-subset/alpha candidates using cross-validation metrics and reports sealed-test metrics only for the selected candidate.


## Load Ridge executions from MLflow

The tracking setup and experiment lookup are initialized here so this section does not depend on variables created by the training cells.


In [ ]:
import json
import re
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from joblib import load as load_joblib

from src.config import (
    CV_SELECTION_METRIC,
    FORECAST_HORIZON_HOURS,
    MLFLOW_TRACKING_URI,
    TARGET_STATION_ID,
    WEATHER_VARIABLES,
)
from src.training import (
    build_feature_subsets,
    load_joined_training_data,
    numeric_predictors,
    prepare_model_rows,
    validate_predictions,
)

if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
RIDGE_EVALUATION_EXPERIMENT_NAME = "ridge"
ridge_experiment = mlflow.get_experiment_by_name(RIDGE_EVALUATION_EXPERIMENT_NAME)
if ridge_experiment is None:
    raise ValueError(
        'No valid Ridge MLflow execution exists: experiment "ridge" was not found. '
        "Run 04_train_ridge.ipynb through its sealed-test cell first."
    )
ridge_runs = mlflow.search_runs(experiment_ids=[ridge_experiment.experiment_id])


def _run_series(runs: pd.DataFrame, column: str) -> pd.Series:
    """Return a run column, or a null series when MLflow has no such field."""
    if column in runs.columns:
        return runs[column]
    return pd.Series(pd.NA, index=runs.index, dtype="object")


def _finite_float(value: object) -> float | None:
    """Convert an MLflow value to a finite float, if possible."""
    if value is None or pd.isna(value):
        return None
    try:
        converted = float(value)
    except (TypeError, ValueError):
        return None
    return converted if np.isfinite(converted) else None


CV_METRIC_COLUMNS = [
    "metrics.cv_mae_mean",
    "metrics.cv_mae_std",
    "metrics.cv_rmse_mean",
    "metrics.cv_rmse_std",
    "metrics.cv_me_mean",
    "metrics.cv_me_std",
    "metrics.cv_r2_mean",
    "metrics.cv_r2_std",
]
CANDIDATE_ID_COLUMNS = [
    "tags.subset",
    "params.alpha",
    "params.feature_count",
]

## Validate and select a completed execution

A finished sealed-test run is the completion marker for one notebook execution. Executions are checked newest first; incomplete candidate sets are skipped in favor of the newest older execution that passes all sanity checks.


In [ ]:
def _validate_ridge_execution(
    sealed_test_run: pd.Series, all_runs: pd.DataFrame
) -> dict[str, object]:
    """Validate one sealed-test marker and assemble its candidate data."""
    execution_uuid = str(sealed_test_run["tags.execution_uuid"])
    execution_candidates = all_runs.loc[
        _run_series(all_runs, "tags.execution_uuid").eq(execution_uuid)
        & _run_series(all_runs, "tags.run_type").eq("candidate_parent")
    ].copy()
    if execution_candidates.empty:
        raise ValueError("candidate-parent rows are missing")

    required_columns = CANDIDATE_ID_COLUMNS + CV_METRIC_COLUMNS
    missing_columns = [
        column
        for column in required_columns
        if column not in execution_candidates.columns
    ]
    if missing_columns:
        raise ValueError(
            f"candidate-parent rows are missing required fields: {missing_columns}"
        )

    candidate_records = []
    candidate_keys = []
    for _, candidate_run in execution_candidates.iterrows():
        subset = candidate_run["tags.subset"]
        alpha = _finite_float(candidate_run["params.alpha"])
        feature_count = _finite_float(candidate_run["params.feature_count"])
        if (
            subset is None
            or pd.isna(subset)
            or not str(subset).strip()
            or alpha is None
            or feature_count is None
            or not feature_count.is_integer()
        ):
            raise ValueError("candidate-parent identifiers are invalid")
        metric_values = {
            metric_column.removeprefix("metrics."): _finite_float(
                candidate_run[metric_column]
            )
            for metric_column in CV_METRIC_COLUMNS
        }
        missing_metrics = [
            metric_name
            for metric_name, metric_value in metric_values.items()
            if metric_value is None
        ]
        if missing_metrics:
            raise ValueError(
                f"candidate-parent CV metrics are missing: {missing_metrics}"
            )
        candidate_key = (str(subset), alpha)
        candidate_keys.append(candidate_key)
        candidate_records.append(
            {
                "subset": str(subset),
                "feature_count": int(feature_count),
                "alpha": alpha,
                **metric_values,
            }
        )

    if len(candidate_keys) != len(set(candidate_keys)):
        raise ValueError("candidate (subset, alpha) keys are not unique")
    candidate_table = (
        pd.DataFrame(candidate_records)
        .sort_values(f"cv_{CV_SELECTION_METRIC}_mean", kind="stable")
        .reset_index(drop=True)
    )

    sealed_subset = sealed_test_run.get("tags.subset")
    sealed_alpha = _finite_float(sealed_test_run.get("params.alpha"))
    if (
        sealed_subset is None
        or pd.isna(sealed_subset)
        or not str(sealed_subset).strip()
        or sealed_alpha is None
    ):
        raise ValueError("sealed-test candidate identifiers are invalid")
    sealed_key = (str(sealed_subset), sealed_alpha)
    if sealed_key not in set(candidate_keys):
        raise ValueError("sealed-test candidate does not match a candidate-parent row")

    sealed_test_metric_columns = [
        "metrics.test_mae",
        "metrics.test_rmse",
        "metrics.test_me",
        "metrics.test_r2",
    ]
    sealed_test_metrics = {
        column.removeprefix("metrics."): _finite_float(sealed_test_run.get(column))
        for column in sealed_test_metric_columns
    }
    missing_test_metrics = [
        metric_name
        for metric_name, metric_value in sealed_test_metrics.items()
        if metric_value is None
    ]
    if missing_test_metrics:
        raise ValueError(f"sealed-test metrics are missing: {missing_test_metrics}")

    horizon_columns: dict[int, dict[str, str]] = {}
    for column in all_runs.columns:
        match = re.fullmatch(
            r"metrics\.(test_mae|test_rmse|test_me|test_r2)_horizon_(\d+)", str(column)
        )
        if match:
            metric_name, horizon = match.groups()
            horizon_columns.setdefault(int(horizon), {})[metric_name] = column
    horizon_records = []
    for horizon in sorted(horizon_columns):
        metric_columns = horizon_columns[horizon]
        if {"test_mae", "test_rmse", "test_me", "test_r2"}.difference(metric_columns):
            continue
        horizon_values = {
            metric_name: _finite_float(sealed_test_run.get(column))
            for metric_name, column in metric_columns.items()
        }
        if any(value is None for value in horizon_values.values()):
            raise ValueError("sealed-test per-horizon metrics are incomplete")
        horizon_records.append({"horizon_hours": horizon, **horizon_values})
    if not horizon_records:
        raise ValueError("sealed-test per-horizon metrics are missing")
    expected_horizons = set(range(1, FORECAST_HORIZON_HOURS + 1))
    if {record["horizon_hours"] for record in horizon_records} != expected_horizons:
        raise ValueError(
            "sealed-test per-horizon metrics do not match the forecast contract"
        )

    selected_candidate = candidate_table.loc[
        candidate_table["subset"].eq(str(sealed_subset))
        & candidate_table["alpha"].eq(sealed_alpha)
    ].iloc[0]
    return {
        "candidate_table": candidate_table,
        "selected_candidate": selected_candidate,
        "sealed_test_run": sealed_test_run,
        "sealed_test_metrics": sealed_test_metrics,
        "horizon_records": horizon_records,
    }

In [ ]:
if ridge_runs.empty:
    raise ValueError(
        "No valid Ridge MLflow execution exists: the Ridge experiment has no runs. "
        "Run 04_train_ridge.ipynb through its sealed-test cell first."
    )
finished_sealed_test_runs = ridge_runs.loc[
    _run_series(ridge_runs, "status").astype("string").str.upper().eq("FINISHED")
    & _run_series(ridge_runs, "tags.run_type").eq("sealed_test")
    & _run_series(ridge_runs, "tags.execution_uuid").notna()
].copy()
if finished_sealed_test_runs.empty:
    raise ValueError(
        "No valid Ridge MLflow execution exists: no FINISHED sealed-test run was found. "
        "Run 04_train_ridge.ipynb through its sealed-test cell first."
    )
finished_sealed_test_runs["_finished_at"] = pd.to_datetime(
    _run_series(finished_sealed_test_runs, "end_time"),
    errors="coerce",
    utc=True,
)
finished_sealed_test_runs = finished_sealed_test_runs.sort_values(
    ["_finished_at", "run_id"],
    ascending=[False, False],
    na_position="last",
).drop_duplicates("tags.execution_uuid", keep="first")

execution_failures = []
selected_execution = None
for _, sealed_test_run in finished_sealed_test_runs.iterrows():
    execution_uuid = str(sealed_test_run["tags.execution_uuid"])
    try:
        selected_execution = _validate_ridge_execution(sealed_test_run, ridge_runs)
    except ValueError as error:
        execution_failures.append(f"{execution_uuid}: {error}")
        continue
    selected_execution_uuid = execution_uuid
    break
if selected_execution is None:
    failure_details = "; ".join(execution_failures)
    raise ValueError(
        "No valid Ridge MLflow execution exists. Each finished sealed-test marker failed "
        f"the candidate sanity checks: {failure_details}. "
        "Run 04_train_ridge.ipynb through a successful sealed-test cell."
    )

selected_candidate_table = selected_execution["candidate_table"]
selected_candidate = selected_execution["selected_candidate"]
selected_sealed_test_run = selected_execution["sealed_test_run"]
selected_sealed_test_metrics = selected_execution["sealed_test_metrics"]
selected_horizon_metrics = pd.DataFrame(selected_execution["horizon_records"])
selected_execution_summary = pd.DataFrame(
    [
        {
            "execution_uuid": selected_execution_uuid,
            "sealed_test_run_id": selected_sealed_test_run.get("run_id"),
            "sealed_test_end_time": selected_sealed_test_run.get("end_time"),
            "candidate_count": len(selected_candidate_table),
            "selected_subset": selected_candidate["subset"],
            "selected_alpha": selected_candidate["alpha"],
        }
    ]
)
display(selected_execution_summary)

## Compare cross-validation candidates

Candidate ranking uses only the parent-run CV metrics and the configured selection metric; the sealed-test run is not used to rank candidates.


In [ ]:
candidate_columns = [
    "subset",
    "feature_count",
    "alpha",
    "cv_mae_mean",
    "cv_mae_std",
    "cv_rmse_mean",
    "cv_rmse_std",
    "cv_me_mean",
    "cv_me_std",
    "cv_r2_mean",
    "cv_r2_std",
]
candidate_comparison_table = selected_candidate_table[candidate_columns].copy()
display(candidate_comparison_table)

## Visualize cross-validation error

The heatmap shows CV MAE across feature subsets and alpha values. The line chart adds fold-to-fold MAE variation as error bars.


In [ ]:
cv_mae_heatmap_values = (
    candidate_comparison_table.pivot(
        index="subset", columns="alpha", values="cv_rmse_mean"
    )
    .sort_index(axis=0)
    .sort_index(axis=1)
)
cv_mae_heatmap_figure = go.Figure(
    data=go.Heatmap(
        z=cv_mae_heatmap_values.to_numpy(),
        x=list(map(str, cv_mae_heatmap_values.columns.tolist())),
        y=cv_mae_heatmap_values.index.tolist(),
        colorbar={"title": "CV RMSE"},
        hovertemplate="Subset=%{y}<br>Alpha=%{x}<br>CV MAE=%{z:.4f}<extra></extra>",
    )
)
cv_mae_heatmap_figure.update_layout(
    title="Ridge candidate CV MAE by feature subset and alpha",
    xaxis_title="Alpha",
    yaxis_title="Feature subset",
)
display(cv_mae_heatmap_figure)

In [ ]:
cv_mae_by_alpha_figure = go.Figure()
for subset_name, subset_candidates in candidate_comparison_table.groupby(
    "subset", sort=True
):
    subset_candidates = subset_candidates.sort_values("alpha")
    cv_mae_by_alpha_figure.add_trace(
        go.Scatter(
            x=subset_candidates["alpha"],
            y=subset_candidates["cv_rmse_mean"],
            mode="lines+markers",
            name=subset_name,
            error_y={
                "type": "data",
                "array": subset_candidates["cv_mae_std"],
                "visible": True,
            },
        )
    )
cv_mae_by_alpha_figure.update_layout(
    title="Ridge CV RMSE versus alpha",
    xaxis_title="Alpha",
    yaxis_title="CV RMSE",
    xaxis_type="log",
)
display(cv_mae_by_alpha_figure)

## Inspect selected-candidate sealed-test performance

These values belong only to the candidate recorded by the selected sealed-test run. The final chart shows its MAE and RMSE at each available forecast horizon.


In [ ]:
selected_candidate_sealed_test_summary = pd.DataFrame(
    [
        {
            "execution_uuid": selected_execution_uuid,
            "subset": selected_candidate["subset"],
            "feature_count": selected_candidate["feature_count"],
            "alpha": selected_candidate["alpha"],
            "cv_mae_mean": selected_candidate["cv_mae_mean"],
            "cv_rmse_mean": selected_candidate["cv_rmse_mean"],
            "cv_me_mean": selected_candidate["cv_me_mean"],
            "cv_r2_mean": selected_candidate["cv_r2_mean"],
            "test_mae": selected_sealed_test_metrics["test_mae"],
            "test_rmse": selected_sealed_test_metrics["test_rmse"],
            "test_me": selected_sealed_test_metrics["test_me"],
            "test_r2": selected_sealed_test_metrics["test_r2"],
        }
    ]
)
display(selected_candidate_sealed_test_summary)

sealed_test_horizon_figure = go.Figure(
    [
        go.Scatter(
            x=selected_horizon_metrics["horizon_hours"],
            y=selected_horizon_metrics[metric_name],
            mode="lines+markers",
            name=metric_name.upper(),
        )
        for metric_name in ("test_mae", "test_rmse", "test_me", "test_r2")
    ]
)
sealed_test_horizon_figure.update_layout(
    title="Selected Ridge candidate sealed-test error by horizon",
    xaxis_title="Forecast horizon (hours)",
    yaxis_title="Error",
)
display(sealed_test_horizon_figure)

## Reload and score the saved Ridge model

Reloads the saved Ridge model and manifest, validates their feature and execution contracts against the selected MLflow candidate, and scores the eligible sealed-test cohort without retraining or changing the stored prediction semantics.


In [ ]:
COMPARISON_PROCESSED_DIR = Path("data/processed/joined")
COMPARISON_METADATA_PATH = (
    COMPARISON_PROCESSED_DIR / "all_stations_feature_metadata.json"
)
COMPARISON_TRAIN_PATH = COMPARISON_PROCESSED_DIR / "all_stations_train_features.parquet"
COMPARISON_TEST_PATH = COMPARISON_PROCESSED_DIR / "all_stations_test_features.parquet"
COMPARISON_MODEL_PATH = Path("models") / f"ridge_{TARGET_STATION_ID}.joblib"
COMPARISON_MODEL_METADATA_PATH = Path("models") / f"ridge_{TARGET_STATION_ID}.json"

(
    comparison_contract,
    _comparison_train_features,
    comparison_test_features,
) = load_joined_training_data(
    COMPARISON_METADATA_PATH,
    COMPARISON_TRAIN_PATH,
    COMPARISON_TEST_PATH,
    station_id=TARGET_STATION_ID,
    forecast_horizon_hours=FORECAST_HORIZON_HOURS,
)
COMPARISON_TARGET_COLUMNS = list(comparison_contract.target_columns)
comparison_feature_subsets = build_feature_subsets(
    comparison_contract,
    weather_variables=WEATHER_VARIABLES,
)
comparison_test_rows = prepare_model_rows(
    comparison_test_features,
    comparison_contract,
    artifact_name="comparison test",
)

if not COMPARISON_MODEL_METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"Missing Ridge model manifest: {COMPARISON_MODEL_METADATA_PATH}"
    )
model_manifest = json.loads(COMPARISON_MODEL_METADATA_PATH.read_text(encoding="utf-8"))

manifest_station_id = model_manifest.get("station_id")
if manifest_station_id != TARGET_STATION_ID:
    raise ValueError(
        "Saved Ridge manifest station_id does not match the current target station"
    )
if model_manifest.get("forecast_horizon_hours") != FORECAST_HORIZON_HOURS:
    raise ValueError(
        "Saved Ridge manifest forecast horizon does not match the current contract"
    )
if tuple(model_manifest.get("full_feature_columns", ())) != tuple(
    comparison_contract.predictor_columns
):
    raise ValueError(
        "Saved Ridge manifest full feature contract does not match the current contract"
    )
if tuple(model_manifest.get("target_columns", ())) != tuple(
    comparison_contract.target_columns
):
    raise ValueError(
        "Saved Ridge manifest target contract does not match the current contract"
    )

manifest_execution_uuid = model_manifest.get("execution_uuid")
if manifest_execution_uuid != selected_execution_uuid:
    raise ValueError(
        "Saved Ridge manifest execution_uuid does not match the selected MLflow execution"
    )
manifest_subset = model_manifest.get("selected_subset")
if manifest_subset != selected_candidate["subset"]:
    raise ValueError(
        "Saved Ridge manifest selected subset does not match the selected MLflow candidate"
    )
if model_manifest.get("feature_subset", manifest_subset) != manifest_subset:
    raise ValueError("Saved Ridge manifest feature subset aliases disagree")
manifest_alpha = _finite_float(model_manifest.get("selected_alpha"))
if manifest_alpha is None or manifest_alpha != float(selected_candidate["alpha"]):
    raise ValueError(
        "Saved Ridge manifest selected alpha does not match the selected MLflow candidate"
    )
comparison_feature_columns = tuple(model_manifest.get("selected_feature_columns", ()))
if not comparison_feature_columns or not set(comparison_feature_columns).issubset(
    set(comparison_contract.predictor_columns)
):
    raise ValueError(
        "Saved Ridge manifest selected features do not match the current contract"
    )
manifest_feature_subsets = model_manifest.get("feature_subsets", {})
if manifest_subset in comparison_feature_subsets:
    if comparison_feature_columns != tuple(comparison_feature_subsets[manifest_subset]):
        raise ValueError(
            "Saved Ridge manifest selected features do not match the selected subset contract"
        )
    if manifest_feature_subsets and tuple(
        manifest_feature_subsets.get(manifest_subset, ())
    ) != tuple(comparison_feature_subsets[manifest_subset]):
        raise ValueError(
            "Saved Ridge manifest feature subset does not match the current contract"
        )

comparison_model = load_joblib(COMPARISON_MODEL_PATH)
comparison_prediction_values = validate_predictions(
    comparison_model.predict(
        numeric_predictors(comparison_test_rows, comparison_feature_columns)
    ),
    expected_rows=len(comparison_test_rows),
    target_columns=COMPARISON_TARGET_COLUMNS,
    artifact_name="saved Ridge sealed-test",
)
print(
    f"Scored {len(comparison_test_rows):,} eligible sealed-test rows with "
    f"{len(COMPARISON_TARGET_COLUMNS)} horizons using the saved Ridge model."
)

In [ ]:
comparison_prediction_columns = [
    f"prediction_{target_column}" for target_column in COMPARISON_TARGET_COLUMNS
]
comparison_prediction_table = (
    comparison_test_rows[["timestamp", *COMPARISON_TARGET_COLUMNS]]
    .reset_index(drop=True)
    .rename(columns={"timestamp": "issue_time"})
)
comparison_prediction_table = pd.concat(
    [
        comparison_prediction_table,
        pd.DataFrame(
            comparison_prediction_values,
            columns=comparison_prediction_columns,
        ),
    ],
    axis=1,
)
comparison_prediction_table["issue_time"] = pd.to_datetime(
    comparison_prediction_table["issue_time"], utc=True
)
# display(comparison_prediction_table)

comparison_issue_times = comparison_prediction_table["issue_time"]
comparison_horizons = list(range(1, FORECAST_HORIZON_HOURS + 1))
comparison_horizon_labels = [f"H+{horizon:02d}" for horizon in comparison_horizons]
comparison_time_series_frames = []
for horizon in comparison_horizons:
    target_column = COMPARISON_TARGET_COLUMNS[horizon - 1]
    prediction_column = comparison_prediction_columns[horizon - 1]
    valid_times = comparison_issue_times + pd.to_timedelta(horizon, unit="h")
    comparison_time_series_frames.append(
        go.Frame(
            name=comparison_horizon_labels[horizon - 1],
            data=[
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[target_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Actual",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Actual=%{y:.3f}<extra></extra>",
                ),
                go.Scattergl(
                    x=valid_times,
                    y=comparison_prediction_table[prediction_column],
                    customdata=comparison_issue_times,
                    mode="lines+markers",
                    name="Prediction",
                    hovertemplate="Valid time=%{x}<br>Issue time=%{customdata}<br>Prediction=%{y:.3f}<extra></extra>",
                ),
            ],
        )
    )
comparison_time_series_steps = [
    {
        "label": comparison_horizon_labels[horizon - 1],
        "method": "animate",
        "args": [[comparison_horizon_labels[horizon - 1]], {"mode": "immediate"}],
    }
    for horizon in comparison_horizons
]
comparison_time_series_figure = go.Figure(
    data=comparison_time_series_frames[0].data,
    frames=comparison_time_series_frames,
    layout={
        "title": "Saved Ridge predictions across forecast horizons",
        "xaxis_title": "Valid time",
        "yaxis_title": "Water level",
        "hovermode": "x unified",
        "sliders": [
            {
                "active": 0,
                "currentvalue": {"prefix": "Forecast horizon: "},
                "steps": comparison_time_series_steps,
            }
        ],
    },
)
ridge_prediction_time_series_figure = comparison_time_series_figure
display(comparison_time_series_figure)

## Inspect best and worst Ridge forecast windows

The following plots use the saved-model predictions and select sealed-test issue times by the RMSE calculated across all 24 forecast horizons. A context window is eligible only when the target-station water-level series contains every hourly observation from 48 hours before through 48 hours after the issue time, with no imputed observations.

In [ ]:
comparison_prediction_table["issue_rmse"] = np.sqrt(
    np.mean(
        (
            comparison_prediction_table[comparison_prediction_columns].to_numpy(
                dtype=float
            )
            - comparison_prediction_table[COMPARISON_TARGET_COLUMNS].to_numpy(
                dtype=float
            )
        )
        ** 2,
        axis=1,
    )
)
if not np.isfinite(comparison_prediction_table["issue_rmse"]).all():
    raise ValueError("Per-issue Ridge RMSE contains non-finite values")

comparison_target_water_level_column = f"{TARGET_STATION_ID}__water_level"
comparison_target_imputed_column = f"{TARGET_STATION_ID}__imputed"
comparison_context_series = pd.concat(
    [
        _comparison_train_features[
            [
                "timestamp",
                comparison_target_water_level_column,
                comparison_target_imputed_column,
            ]
        ],
        comparison_test_features[
            [
                "timestamp",
                comparison_target_water_level_column,
                comparison_target_imputed_column,
            ]
        ],
    ],
    ignore_index=True,
)
comparison_context_series["timestamp"] = pd.to_datetime(
    comparison_context_series["timestamp"], utc=True
)
comparison_context_series = (
    comparison_context_series.sort_values("timestamp")
    .drop_duplicates("timestamp", keep="last")
    .set_index("timestamp")
    .sort_index()
)


def _complete_comparison_context(
    issue_time: pd.Timestamp,
) -> pd.DataFrame | None:
    """Return a complete, non-imputed +/-48-hour context if available."""
    issue_time = pd.Timestamp(issue_time)
    if issue_time.tz is None:
        issue_time = issue_time.tz_localize("UTC")
    else:
        issue_time = issue_time.tz_convert("UTC")
    expected_times = pd.date_range(
        issue_time - pd.to_timedelta(48, unit="h"),
        issue_time + pd.to_timedelta(48, unit="h"),
        freq="h",
    )
    context = comparison_context_series.reindex(expected_times)
    if (
        len(context) != 97
        or not context[comparison_target_water_level_column].notna().all()
        or not context[comparison_target_imputed_column].eq(False).all()
    ):
        return None
    return context


comparison_forecast_window_context_by_row = {}
for row_index, prediction_row in comparison_prediction_table.iterrows():
    context = _complete_comparison_context(prediction_row["issue_time"])
    if context is not None:
        comparison_forecast_window_context_by_row[row_index] = context
if len(comparison_forecast_window_context_by_row) < 2:
    raise ValueError(
        "Ridge forecast-window plots require at least two sealed-test issue timestamps "
        "with complete, non-imputed target-station context from -48h through +48h; "
        f"found {len(comparison_forecast_window_context_by_row)}."
    )

comparison_forecast_window_candidates = comparison_prediction_table.loc[
    list(comparison_forecast_window_context_by_row)
].sort_values(["issue_rmse", "issue_time"], kind="stable")
best_forecast_window_row = comparison_forecast_window_candidates.iloc[0]
worst_forecast_window_candidates = comparison_prediction_table.loc[
    list(comparison_forecast_window_context_by_row)
].sort_values(
    ["issue_rmse", "issue_time"],
    ascending=[False, True],
    kind="stable",
)
worst_forecast_window_row = worst_forecast_window_candidates.iloc[0]


def _comparison_forecast_window_slider_rows(
    window_row: pd.Series,
) -> pd.DataFrame:
    """Return eligible issue times within +/-12 hours of a chart issue."""
    issue_time = pd.Timestamp(window_row["issue_time"])
    window_start = issue_time - pd.to_timedelta(12, unit="h")
    window_end = issue_time + pd.to_timedelta(12, unit="h")
    return (
        comparison_prediction_table.loc[list(comparison_forecast_window_context_by_row)]
        .loc[lambda rows: rows["issue_time"].between(window_start, window_end)]
        .sort_values("issue_time", kind="stable")
    )


def _comparison_forecast_window_traces(
    window_row: pd.Series,
    context: pd.DataFrame,
) -> list[go.Scatter]:
    """Build ground-truth and 24-hour prediction traces for one issue."""
    return [
        go.Scatter(
            x=context.index,
            y=context[comparison_target_water_level_column].to_numpy(dtype=float),
            mode="lines+markers",
            name="Ground truth",
            hovertemplate="Valid time=%{x}<br>Ground truth=%{y:.3f}<extra></extra>",
        ),
        _comparison_forecast_window_prediction_trace(window_row),
    ]


def _comparison_forecast_window_prediction_trace(
    window_row: pd.Series,
) -> go.Scatter:
    """Build the issue-specific 24-hour prediction trace."""
    issue_time = pd.Timestamp(window_row["issue_time"])
    prediction_times = issue_time + pd.to_timedelta(comparison_horizons, unit="h")
    return go.Scatter(
        x=prediction_times,
        y=window_row[comparison_prediction_columns].to_numpy(dtype=float),
        mode="lines+markers",
        name="Prediction",
        hovertemplate="Valid time=%{x}<br>Prediction=%{y:.3f}<extra></extra>",
    )


def _comparison_forecast_window_layout(
    window_row: pd.Series,
    label: str,
) -> dict:
    """Build the issue-specific title and marker annotation."""
    issue_time = pd.Timestamp(window_row["issue_time"])
    return {
        "title": (
            f"{label} Ridge forecast window<br>"
            f"Issue time: {issue_time.isoformat()} | "
            f"24-hour RMSE: {window_row['issue_rmse']:.4f}"
        ),
        "xaxis_title": "Valid time",
        "yaxis_title": "Water level",
        "hovermode": "x unified",
        "margin": {"b": 160},
        "shapes": [
            {
                "type": "line",
                "x0": issue_time,
                "x1": issue_time,
                "y0": 0,
                "y1": 1,
                "yref": "paper",
                "line": {"dash": "dash", "color": "black"},
            }
        ],
        "annotations": [
            {
                "x": issue_time,
                "y": 1,
                "yref": "paper",
                "text": "Issue time",
                "showarrow": True,
                "arrowhead": 2,
            }
        ],
    }


def _build_comparison_forecast_window_figure(
    window_row: pd.Series,
    context: pd.DataFrame,
    label: str,
) -> go.Figure:
    """Build an issue-time forecast chart with a +/-12-hour slider.

    The selected issue's ground-truth context remains fixed while the slider
    updates the prediction trace and issue-time annotations.
    """
    slider_rows = _comparison_forecast_window_slider_rows(window_row)
    slider_row_indices = slider_rows.index.tolist()
    active_slider_index = slider_row_indices.index(window_row.name)
    frames = []
    slider_steps = []
    for row_index, slider_row in slider_rows.iterrows():
        frame_name = f"forecast-window-{row_index}"
        frames.append(
            go.Frame(
                name=frame_name,
                data=[_comparison_forecast_window_prediction_trace(slider_row)],
                traces=[1],
                layout=_comparison_forecast_window_layout(slider_row, label),
            )
        )
        slider_steps.append(
            {
                "label": pd.Timestamp(slider_row["issue_time"]).strftime(
                    "%Y-%m-%d %H:%M UTC"
                ),
                "method": "animate",
                "args": [
                    [frame_name],
                    {
                        "mode": "immediate",
                        "frame": {"duration": 0, "redraw": True},
                        "transition": {"duration": 0},
                    },
                ],
            }
        )
    figure = go.Figure(
        data=_comparison_forecast_window_traces(window_row, context),
        frames=frames,
        layout={
            **_comparison_forecast_window_layout(window_row, label),
            "sliders": [
                {
                    "active": active_slider_index,
                    "y": -0.28,
                    "pad": {"t": 12},
                    "currentvalue": {"prefix": "Issue time: "},
                    "steps": slider_steps,
                }
            ],
        },
    )
    return figure

### Best 

In [ ]:
best_ridge_forecast_window_figure = _build_comparison_forecast_window_figure(
    best_forecast_window_row,
    comparison_forecast_window_context_by_row[best_forecast_window_row.name],
    "Best-RMSE",
)
display(best_ridge_forecast_window_figure)

### Worst

In [ ]:
worst_ridge_forecast_window_figure = _build_comparison_forecast_window_figure(
    worst_forecast_window_row,
    comparison_forecast_window_context_by_row[worst_forecast_window_row.name],
    "Worst-RMSE",
)
display(worst_ridge_forecast_window_figure)

## Compare absolute and signed errors

Each box contains all eligible sealed-test errors for one horizon. Absolute-error markers reuse the selected MLflow MAE/RMSE values; signed errors follow the convention `prediction - actual`.


In [ ]:
comparison_actual_values = comparison_test_rows[COMPARISON_TARGET_COLUMNS].to_numpy(
    dtype=float
)
signed_errors = comparison_prediction_values - comparison_actual_values
absolute_errors = np.abs(signed_errors)

absolute_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(absolute_errors),
            y=absolute_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_mae"],
            mode="markers",
            name="MAE",
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=selected_horizon_metrics["test_rmse"],
            mode="markers",
            name="RMSE",
        ),
    ]
)
absolute_error_boxplot_figure.update_layout(
    title="Saved Ridge absolute errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Absolute error",
)

display(absolute_error_boxplot_figure)

In [ ]:
signed_error_boxplot_figure = go.Figure(
    data=[
        go.Box(
            x=comparison_horizon_labels * len(signed_errors),
            y=signed_errors.reshape(-1),
            name="Boxplots",
            boxpoints=False,
        ),
        go.Scatter(
            x=comparison_horizon_labels,
            y=signed_errors.mean(axis=0),
            mode="markers",
            name="Mean error",
        ),
    ]
)
signed_error_boxplot_figure.update_layout(
    title="Saved Ridge signed errors by forecast horizon",
    xaxis={
        "title": "Forecast horizon",
        "type": "category",
        "categoryorder": "array",
        "categoryarray": comparison_horizon_labels,
    },
    yaxis_title="Signed error (prediction - actual)",
)
signed_error_boxplot_figure.add_hline(
    y=0,
    line_dash="dash",
    line_color="black",
)
display(signed_error_boxplot_figure)